# Clasificador de plagas y enfermedades — AgroTech Hidalgo

Entrena una red convolucional que identifica plagas y enfermedades a partir de una
foto, y la exporta a **TensorFlow.js** para que corra dentro del navegador del
agricultor, **sin internet y sin costo por consulta**.

## Qué necesitas antes de empezar

1. **Cuenta de Kaggle** (gratis) → https://www.kaggle.com
2. **Token de la API de Kaggle**: en Kaggle entra a `Settings → API → Create New Token`.
   Kaggle entrega uno de dos formatos según la versión: un **token** que empieza
   con `KGAT_...`, o un archivo **`kaggle.json`**. La celda 3 acepta los dos.
3. **GPU activada en Colab**: menú `Entorno de ejecución → Cambiar tipo de entorno de
   ejecución → Acelerador por hardware: GPU`. Sin GPU esto tarda muchísimo.

## Cuánto tarda

| Etapa | Tiempo aproximado |
|---|---|
| Descarga de los dos datasets | 15–30 min |
| Preparación y unificación | 5–10 min |
| Entrenamiento fase 1 (cabeza) | 20–40 min |
| Entrenamiento fase 2 (ajuste fino) | 40–90 min |
| Exportación | 2–5 min |

Colab gratuito desconecta por inactividad: deja la pestaña abierta y vuelve a
revisar cada tanto. Los checkpoints se guardan, así que si se corta puedes
reanudar sin perder todo.

## Advertencia sobre los datos — léela

Los dos datasets **no son igual de difíciles**:

- **PlantVillage** son hojas fotografiadas en laboratorio, sobre fondo liso y
  uniforme. Los modelos alcanzan cifras altísimas ahí, pero **caen bastante con
  fotos reales de campo**, porque aprenden parte del fondo y de la iluminación.
- **IP102** son insectos fotografiados en condiciones reales. Es un problema de
  grano fino y genuinamente difícil; los resultados publicados son mucho más
  modestos.

Si mezclas los dos y reportas una sola cifra, el promedio queda inflado por la
parte fácil. Por eso este notebook **evalúa y reporta cada dataset por separado**.
Presenta ambos números: es más honesto y se defiende mejor.

## 1. Comprobar la GPU

In [ ]:
import tensorflow as tf

print('TensorFlow:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('GPU detectada:', gpus[0].name)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('SIN GPU. Ve a Entorno de ejecucion -> Cambiar tipo -> Acelerador: GPU')
    print('Puedes continuar, pero el entrenamiento tardara horas.')

## 2. Instalar el conversor a TensorFlow.js

`tensorflowjs` es lo que traduce el modelo entrenado al formato que entiende el
navegador. Colab pedirá reiniciar el entorno: **acepta**, y luego continúa desde
la celda 3 (no repitas la 1 y la 2).

In [ ]:
!pip install -q tensorflowjs
print('Listo. Si Colab pide reiniciar el entorno, acepta y sigue en la celda 3.')

## 3. Credenciales de Kaggle

La celda detecta qué formato de credencial tienes y te guía:

- **Token nuevo** (`KGAT_...`): lo pegas cuando te lo pida. Se escribe **oculto**,
  a propósito — la salida de las celdas queda guardada dentro del `.ipynb`, así que
  un token escrito a la vista viajaría dentro del archivo si lo compartes o lo
  subes a git.
- **Archivo `kaggle.json`** (formato anterior): lo subes con el selector.

Las credenciales viven solo en esta sesión de Colab y desaparecen cuando el
entorno se recicla. Cuando termines el entrenamiento, revoca el token en Kaggle:
ya no lo necesitas y un token que no existe no se puede filtrar.

In [ ]:
import os, getpass, subprocess
from pathlib import Path

# Kaggle tiene DOS formatos de credencial y este notebook acepta los dos:
#   nuevo  -> un token que empieza con KGAT_, va en ~/.kaggle/access_token
#   viejo  -> un archivo kaggle.json, va en ~/.kaggle/kaggle.json
DIR_KAGGLE = Path.home() / '.kaggle'
DIR_KAGGLE.mkdir(parents=True, exist_ok=True)

ruta_token = DIR_KAGGLE / 'access_token'
ruta_json = DIR_KAGGLE / 'kaggle.json'

# El cliente antiguo NO entiende el token nuevo: hay que actualizarlo siempre.
!pip install -q --upgrade kaggle

if ruta_token.exists() or ruta_json.exists():
    print('Ya hay credenciales cargadas en esta sesion.')
else:
    print('Que tipo de credencial tienes?')
    print('  1) Token nuevo, empieza con KGAT_   (Kaggle -> Settings -> API)')
    print('  2) Archivo kaggle.json              (formato anterior)')
    opcion = input('Escribe 1 o 2: ').strip()

    if opcion == '1':
        # getpass evita que el token se imprima en pantalla. Esto IMPORTA: la
        # salida de las celdas queda guardada DENTRO del .ipynb, asi que un
        # token escrito con input() viajaria dentro del archivo si lo compartes
        # o lo subes a git.
        token = getpass.getpass('Pega el token (no se vera mientras escribes): ').strip()
        if not token.startswith('KGAT_'):
            raise SystemExit('Eso no parece un token de Kaggle: debe empezar con KGAT_')
        ruta_token.write_text(token)
        os.chmod(ruta_token, 0o600)
        os.environ['KAGGLE_API_TOKEN'] = token
        print('Token guardado en ~/.kaggle/access_token')
    else:
        from google.colab import files
        print('Selecciona tu kaggle.json:')
        subidos = files.upload()
        nombre = next((n for n in subidos if n.endswith('.json')), None)
        if nombre is None:
            raise SystemExit('No subiste un .json. Vuelve a ejecutar la celda.')
        ruta_json.write_bytes(subidos[nombre])
        os.chmod(ruta_json, 0o600)
        print('Credenciales guardadas en ~/.kaggle/kaggle.json')

# Comprobacion temprana: si la autenticacion falla, mejor enterarse ahora y no
# a la mitad de una descarga de varios gigabytes.
#
# Se usa 'datasets list -s' a secas porque las opciones del cliente de Kaggle
# cambian entre versiones. Un fallo de sintaxis tambien devuelve codigo distinto
# de cero, asi que hay que distinguirlo de un fallo de credencial: si no, una
# opcion no reconocida se reporta como 'token invalido' y manda a revisar donde
# no hay nada roto.
print('\nVerificando el acceso a Kaggle...')
r = subprocess.run(['kaggle', 'datasets', 'list', '-s', 'plantvillage'],
                   capture_output=True, text=True)
salida = (r.stdout or '') + (r.stderr or '')

if r.returncode == 0:
    print(salida[:400])
    print('OK: la credencial funciona.')
elif 'unrecognized arguments' in salida or salida.lstrip().startswith('usage:'):
    print('Aviso: no se pudo verificar por diferencias de version del cliente.')
    print('NO es un problema de credencial. Continua con la celda 4.')
elif '401' in salida or '403' in salida or 'nauthorized' in salida or 'orbidden' in salida:
    print(salida[:400])
    print('FALLO DE CREDENCIAL. El token no es valido o expiro. Genera uno nuevo')
    print('en Kaggle -> Settings -> API y vuelve a ejecutar esta celda.')
else:
    print(salida[:400])
    print('No se pudo verificar, pero puede ser algo pasajero de la red.')
    print('Intenta la celda 4: si la descarga arranca, la credencial esta bien.')

## 4. Descargar los datasets

Se intentan varios identificadores por dataset porque los mirrors de Kaggle
cambian de nombre y desaparecen con el tiempo. Si todos fallan, busca en Kaggle
"PlantVillage" o "IP102" y sustituye el identificador en la lista.

In [ ]:
import subprocess, shutil
from pathlib import Path

RAIZ = Path('/content/datos')
RAIZ.mkdir(parents=True, exist_ok=True)

# Varios candidatos por dataset: se prueba en orden hasta que uno funcione.
CANDIDATOS = {
    'plantvillage': [
        'abdallahalidev/plantvillage-dataset',
        'emmarex/plantdisease',
        'vipoooool/new-plant-diseases-dataset',
    ],
    'ip102': [
        'rtlmhjbn/ip02-dataset',
        'shubhamsharma170/ip102-dataset',
    ],
}


def descargar(nombre, identificadores):
    destino = RAIZ / nombre
    if destino.exists() and any(destino.iterdir()):
        print(f'[{nombre}] ya descargado en {destino}')
        return destino

    destino.mkdir(parents=True, exist_ok=True)
    for ident in identificadores:
        print(f'[{nombre}] intentando {ident} ...')
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', ident, '-p', str(destino), '--unzip'],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and any(destino.iterdir()):
            print(f'[{nombre}] descargado desde {ident}')
            return destino
        print(f'   fallo: {r.stderr.strip()[:300]}')

    raise SystemExit(
        f'No se pudo descargar "{nombre}". Busca el dataset en Kaggle, acepta sus '
        f'terminos si los pide, y agrega el identificador a CANDIDATOS.'
    )


dir_pv = descargar('plantvillage', CANDIDATOS['plantvillage'])
dir_ip = descargar('ip102', CANDIDATOS['ip102'])

!du -sh /content/datos/*

## 5. Descubrir la estructura real de cada dataset

No se asume ninguna ruta fija. Se recorre el árbol buscando el directorio que
contiene más subcarpetas con imágenes dentro: ese es el que agrupa las clases.
Así el notebook sobrevive a que el mirror de Kaggle cambie su organización.

In [ ]:
EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG'}


def cuenta_imagenes(carpeta, tope=40):
    """Cuenta imagenes directas de una carpeta, cortando pronto por velocidad."""
    n = 0
    try:
        for f in carpeta.iterdir():
            if f.is_file() and f.suffix in EXT:
                n += 1
                if n >= tope:
                    break
    except (PermissionError, OSError):
        pass
    return n


def buscar_raiz_de_clases(base, min_clases=5):
    """Devuelve el directorio con mas subcarpetas que contienen imagenes."""
    mejor, mejor_n = None, 0

    for actual, subdirs, _ in os.walk(base):
        p = Path(actual)
        # Evita descender dentro de las propias carpetas de clase.
        if p.name.lower() in {'train', 'val', 'test', 'valid'} and mejor is not None:
            pass
        con_imagenes = sum(1 for d in subdirs if cuenta_imagenes(p / d) > 0)
        if con_imagenes > mejor_n:
            mejor, mejor_n = p, con_imagenes

    if mejor is None or mejor_n < min_clases:
        raise SystemExit(f'No encontre carpetas de clase dentro de {base}')

    return mejor, mejor_n


raiz_pv, n_pv = buscar_raiz_de_clases(dir_pv)
raiz_ip, n_ip = buscar_raiz_de_clases(dir_ip)

# PlantVillage suele traer color/, grayscale/ y segmented/. Queremos color.
for variante in ['color', 'Color']:
    cand = raiz_pv.parent / variante
    if cand.exists() and cand.is_dir():
        c, n = buscar_raiz_de_clases(cand)
        raiz_pv, n_pv = c, n
        print(f'Usando la variante en color de PlantVillage.')
        break

print(f'PlantVillage -> {raiz_pv}  ({n_pv} clases)')
print(f'IP102        -> {raiz_ip}  ({n_ip} clases)')
print()
print('Ejemplos de clase en PlantVillage:')
for d in sorted([d for d in raiz_pv.iterdir() if d.is_dir()])[:6]:
    print('  ', d.name)
print('Ejemplos de clase en IP102:')
for d in sorted([d for d in raiz_ip.iterdir() if d.is_dir()])[:6]:
    print('  ', d.name)

## 6. Unificar los dos datasets

Se construye un solo directorio con una carpeta por clase. Cada clase lleva
prefijo `pv_` o `ip_` para saber de qué dataset viene y poder evaluarlas por
separado después.

Se usan **enlaces simbólicos** en vez de copiar: son ~130 000 archivos y copiarlos
llenaría el disco de Colab y tardaría muchísimo.

Las clases con muy pocas imágenes se descartan: no alcanzan para aprender nada
útil y solo ensucian las métricas.

In [ ]:
import random
from collections import Counter

UNIFICADO = Path('/content/unificado')
MIN_POR_CLASE = 80   # clases con menos imagenes que esto se descartan
MAX_POR_CLASE = 1500 # tope para que las clases enormes no dominen

random.seed(42)

if UNIFICADO.exists():
    shutil.rmtree(UNIFICADO)
UNIFICADO.mkdir(parents=True)


def recolectar(raiz):
    """{nombre_clase: [rutas]} recorriendo recursivamente cada carpeta de clase."""
    clases = {}
    for d in sorted(raiz.iterdir()):
        if not d.is_dir():
            continue
        imgs = [p for p in d.rglob('*') if p.is_file() and p.suffix in EXT]
        if imgs:
            clases[d.name] = imgs
    return clases


def enlazar(clases, prefijo, resumen):
    for nombre, imgs in clases.items():
        if len(imgs) < MIN_POR_CLASE:
            resumen['descartadas'].append((f'{prefijo}{nombre}', len(imgs)))
            continue

        if len(imgs) > MAX_POR_CLASE:
            imgs = random.sample(imgs, MAX_POR_CLASE)

        destino = UNIFICADO / f'{prefijo}{nombre}'
        destino.mkdir(parents=True, exist_ok=True)
        for i, src in enumerate(imgs):
            try:
                os.symlink(src, destino / f'{i:05d}{src.suffix.lower()}')
            except FileExistsError:
                pass
        resumen['aceptadas'].append((f'{prefijo}{nombre}', len(imgs)))


resumen = {'aceptadas': [], 'descartadas': []}
enlazar(recolectar(raiz_pv), 'pv_', resumen)
enlazar(recolectar(raiz_ip), 'ip_', resumen)

total = sum(n for _, n in resumen['aceptadas'])
n_pv_ok = sum(1 for c, _ in resumen['aceptadas'] if c.startswith('pv_'))
n_ip_ok = sum(1 for c, _ in resumen['aceptadas'] if c.startswith('ip_'))

print(f'Clases aceptadas: {len(resumen["aceptadas"])}  ({n_pv_ok} de PlantVillage, {n_ip_ok} de IP102)')
print(f'Imagenes totales: {total:,}')
print(f'Clases descartadas por tener menos de {MIN_POR_CLASE} imagenes: {len(resumen["descartadas"])}')
for c, n in resumen['descartadas'][:10]:
    print(f'   {c}: {n}')

conteos = sorted(resumen['aceptadas'], key=lambda x: x[1])
print(f'\nClase mas pequena: {conteos[0][0]} ({conteos[0][1]})')
print(f'Clase mas grande:  {conteos[-1][0]} ({conteos[-1][1]})')

## 7. Mapear las clases al catálogo de la app

Aquí se conecta el clasificador con `src/lib/plagas/catalogo.ts`. Cuando el
modelo predice una clase que corresponde a una plaga con ficha, la app puede
mostrar ciclo biológico, señales en campo y estrategias de control. Si no hay
ficha, muestra solo el nombre.

El mapeo es por palabras clave sobre el nombre real de la carpeta, y **se imprime
para que lo revises**. Los nombres de clase varían entre mirrors: si ves algo mal
asignado, corrige `PATRONES` y vuelve a ejecutar la celda.

In [ ]:
import re

# id del catalogo -> lista de expresiones que deben aparecer en el nombre de clase.
# El id DEBE existir en src/lib/plagas/catalogo.ts o la app no encontrara la ficha.
PATRONES = {
    'tizon-tardio':        [r'late.?blight', r'phytophthora', r'tizon'],
    'arana-roja':          [r'spider.?mite', r'tetranychus', r'red.?spider'],
    'cenicilla-cereales':  [r'powdery.?mildew', r'oidium', r'cenicilla'],
    'roya-amarilla':       [r'rust', r'puccinia', r'roya'],
    'gusano-cogollero':    [r'fall.?army.?worm', r'spodoptera', r'cogollero', r'army.?worm'],
    'pulgon-del-maiz':     [r'aphid', r'pulgon', r'rhopalosiphum'],
    'mosquita-blanca':     [r'white.?fly', r'whitefly', r'bemisia', r'mosca.?blanca'],
    'trips':               [r'thrip', r'frankliniella'],
    'chapulin':            [r'grasshopper', r'locust', r'chapulin', r'sphenarium'],
    'gusano-trozador':     [r'cutworm', r'agrotis', r'trozador'],
    'gallina-ciega':       [r'grub', r'wireworm', r'phyllophaga', r'gallina.?ciega'],
    'pudricion-radicular': [r'fusarium', r'root.?rot', r'pudricion'],
}

clases = sorted(d.name for d in UNIFICADO.iterdir() if d.is_dir())


def mapear(nombre_clase):
    limpio = nombre_clase.lower().replace('_', ' ').replace('-', ' ')
    for plaga_id, patrones in PATRONES.items():
        if any(re.search(p, limpio) for p in patrones):
            return plaga_id
    return None


def etiqueta_legible(nombre_clase):
    """'pv_Tomato___Late_blight' -> 'Tomato - Late blight'"""
    s = re.sub(r'^(pv|ip)_', '', nombre_clase)
    s = s.replace('___', ' - ').replace('__', ' - ').replace('_', ' ')
    return re.sub(r'\s+', ' ', s).strip()


MAPEO = [
    {
        'indice': i,
        'clase': c,
        'etiqueta': etiqueta_legible(c),
        'origen': 'plantvillage' if c.startswith('pv_') else 'ip102',
        'plagaId': mapear(c),
    }
    for i, c in enumerate(clases)
]

con_ficha = [m for m in MAPEO if m['plagaId']]
print(f'{len(clases)} clases en total; {len(con_ficha)} enlazadas al catalogo.\n')
print('REVISA ESTAS ASIGNACIONES:')
for m in con_ficha:
    print(f"  {m['clase']:<58} -> {m['plagaId']}")

sin_ficha = [m['clase'] for m in MAPEO if not m['plagaId']]
print(f'\nSin ficha en el catalogo ({len(sin_ficha)}), mostrara solo el nombre:')
for c in sin_ficha[:15]:
    print('  ', c)
if len(sin_ficha) > 15:
    print(f'   ... y {len(sin_ficha) - 15} mas')

## 8. Pipeline de datos

División 80/10/10 con semilla fija para que sea reproducible.

El aumento de datos es deliberadamente agresivo (giros, rotación, zoom, contraste,
brillo). Es lo que reduce el problema del fondo uniforme de PlantVillage: obliga
al modelo a fijarse en la lesión y no en la iluminación del estudio.

In [ ]:
TAM = 224          # MobileNetV3 espera 224x224
LOTE = 32
SEMILLA = 42

comun = dict(
    directory=str(UNIFICADO),
    labels='inferred',
    label_mode='int',
    class_names=clases,
    image_size=(TAM, TAM),
    batch_size=LOTE,
    seed=SEMILLA,
    follow_links=True,   # imprescindible: el dataset unificado son symlinks
)

ds_train = tf.keras.utils.image_dataset_from_directory(
    **comun, validation_split=0.2, subset='training')
ds_resto = tf.keras.utils.image_dataset_from_directory(
    **comun, validation_split=0.2, subset='validation')

# El 20 % restante se parte a la mitad: validacion y prueba.
lotes_resto = tf.data.experimental.cardinality(ds_resto).numpy()
ds_val = ds_resto.take(lotes_resto // 2)
ds_test = ds_resto.skip(lotes_resto // 2)

N_CLASES = len(clases)
print(f'{N_CLASES} clases')
print(f'Lotes -> entrenamiento: {tf.data.experimental.cardinality(ds_train).numpy()}, '
      f'validacion: {tf.data.experimental.cardinality(ds_val).numpy()}, '
      f'prueba: {tf.data.experimental.cardinality(ds_test).numpy()}')

aumento = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name='aumento')

AUTO = tf.data.AUTOTUNE
ds_train_p = (ds_train
              .map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTO)
              .prefetch(AUTO))
ds_val_p = ds_val.prefetch(AUTO)
ds_test_p = ds_test.prefetch(AUTO)

## 9. Pesos por clase

Las clases están desbalanceadas: unas tienen 1500 imágenes y otras 80. Sin
corregirlo, el modelo aprende a ignorar las minoritarias y aun así consigue buena
exactitud global — una trampa clásica. Los pesos hacen que equivocarse en una
clase pequeña cueste más.

In [ ]:
import numpy as np

conteo = np.array([
    len([p for p in (UNIFICADO / c).iterdir() if p.suffix.lower() in {e.lower() for e in EXT}])
    for c in clases
], dtype=np.float64)

pesos = conteo.sum() / (len(clases) * np.maximum(conteo, 1))
pesos = np.clip(pesos, 0.2, 6.0)  # sin topes, una clase diminuta desestabiliza el entrenamiento
PESOS_CLASE = {i: float(w) for i, w in enumerate(pesos)}

print(f'Imagenes por clase -> min {int(conteo.min())}, max {int(conteo.max())}, '
      f'mediana {int(np.median(conteo))}')
print(f'Pesos -> min {pesos.min():.2f}, max {pesos.max():.2f}')

## 10. El modelo

**MobileNetV3-Large** preentrenada en ImageNet. Se eligió por tamaño: el modelo
exportado ronda unos pocos megabytes, que es lo que hace viable descargarlo una
vez en un celular con conexión rural y luego usarlo sin datos.

El respaldo (backbone) se congela primero: la cabeza nueva empieza con pesos
aleatorios y, si se entrenara todo junto, sus gradientes destrozarían lo que la
red ya sabe ver.

In [ ]:
base = tf.keras.applications.MobileNetV3Large(
    input_shape=(TAM, TAM, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True,  # normaliza de 0-255 internamente
)
base.trainable = False

entradas = tf.keras.Input(shape=(TAM, TAM, 3), name='imagen')
x = base(entradas, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
salidas = tf.keras.layers.Dense(N_CLASES, activation='softmax', name='plaga')(x)

modelo = tf.keras.Model(entradas, salidas, name='clasificador_plagas')

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3')],
)

print(f'Parametros totales:    {modelo.count_params():,}')
print(f'Parametros entrenables: {sum(np.prod(v.shape) for v in modelo.trainable_variables):,}')

## 11. Fase 1 — entrenar solo la cabeza

Rápida, porque el 99 % de la red está congelada. Si Colab se desconecta, los
checkpoints quedan en `/content/checkpoints` y puedes reanudar.

In [ ]:
CKPT = Path('/content/checkpoints')
CKPT.mkdir(exist_ok=True)

callbacks_f1 = [
    tf.keras.callbacks.ModelCheckpoint(
        str(CKPT / 'fase1.keras'), save_best_only=True, monitor='val_accuracy', mode='max'),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True, mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
]

hist1 = modelo.fit(
    ds_train_p,
    validation_data=ds_val_p,
    epochs=15,
    class_weight=PESOS_CLASE,
    callbacks=callbacks_f1,
)

## 12. Fase 2 — ajuste fino

Se descongela el tercio superior del backbone con una tasa de aprendizaje 50
veces menor. Ahí es donde el modelo pasa de reconocer "texturas de ImageNet" a
reconocer lesiones foliares y morfología de insectos.

Las capas de BatchNormalization se dejan congeladas a propósito: descongelarlas
con lotes pequeños desestabiliza el entrenamiento.

In [ ]:
base.trainable = True

corte = int(len(base.layers) * 0.66)
for capa in base.layers[:corte]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, tf.keras.layers.BatchNormalization):
        capa.trainable = False

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3')],
)

print(f'Capas entrenables: {sum(1 for c in base.layers if c.trainable)} de {len(base.layers)}')

callbacks_f2 = [
    tf.keras.callbacks.ModelCheckpoint(
        str(CKPT / 'mejor.keras'), save_best_only=True, monitor='val_accuracy', mode='max'),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7),
]

hist2 = modelo.fit(
    ds_train_p,
    validation_data=ds_val_p,
    epochs=30,
    class_weight=PESOS_CLASE,
    callbacks=callbacks_f2,
)

## 13. Evaluación honesta — por dataset separado

**Esta es la celda importante para tu tesis y para el jurado.**

La exactitud global mezcla la parte fácil con la difícil y sale inflada. Aquí se
separa: cuánto acierta en PlantVillage (laboratorio) y cuánto en IP102 (campo).
Reporta los dos. Esperar que el segundo sea claramente menor es lo normal, no un
fallo tuyo — es la naturaleza del problema.

In [ ]:
print('Evaluando sobre el conjunto de prueba...')

y_true, y_pred, y_conf = [], [], []
for lote_x, lote_y in ds_test_p:
    probs = modelo.predict(lote_x, verbose=0)
    y_true.extend(lote_y.numpy())
    y_pred.extend(probs.argmax(axis=1))
    y_conf.extend(probs.max(axis=1))

y_true = np.array(y_true); y_pred = np.array(y_pred); y_conf = np.array(y_conf)

origen = np.array(['plantvillage' if clases[i].startswith('pv_') else 'ip102' for i in y_true])
acierto = (y_true == y_pred)

print('\n' + '=' * 62)
print('RESULTADOS')
print('=' * 62)
print(f'Global                    : {acierto.mean():.1%}  ({len(y_true):,} imagenes)')

for etq, nombre in [('plantvillage', 'PlantVillage (laboratorio)'),
                    ('ip102', 'IP102 (campo real)')]:
    m = origen == etq
    if m.sum():
        print(f'{nombre:<26}: {acierto[m].mean():.1%}  ({m.sum():,} imagenes)')

print('=' * 62)
print('\nReporta SIEMPRE los tres numeros. El global por si solo enganya.')

# Umbral de confianza: por debajo, la app debe decir "no estoy seguro".
print('\nEfecto del umbral de confianza:')
print(f"{'umbral':>8} {'cobertura':>11} {'acierto':>9}")
for u in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = y_conf >= u
    if m.sum():
        print(f'{u:>8.0%} {m.mean():>11.1%} {acierto[m].mean():>9.1%}')
print('\nSube el umbral hasta que el acierto te convenza; la cobertura que pierdas')
print('son fotos donde el modelo dira "no estoy seguro", que es lo correcto.')

# Las peores clases indican donde hace falta mas datos.
print('\nClases con peor desempenyo (ahi conviene reunir mas fotos):')
por_clase = []
for i, c in enumerate(clases):
    m = y_true == i
    if m.sum() >= 5:
        por_clase.append((acierto[m].mean(), c, int(m.sum())))
for acc, c, n in sorted(por_clase)[:12]:
    print(f'  {acc:>6.1%}  {c}  (n={n})')

## 14. Exportar a TensorFlow.js

In [ ]:
import tensorflowjs as tfjs

SALIDA = Path('/content/modelo_web')
if SALIDA.exists():
    shutil.rmtree(SALIDA)
SALIDA.mkdir(parents=True)

DIR_MODELO = SALIDA / 'modelo'
tfjs.converters.save_keras_model(modelo, str(DIR_MODELO))

tam_mb = sum(f.stat().st_size for f in DIR_MODELO.rglob('*') if f.is_file()) / 1e6
print(f'Modelo exportado: {tam_mb:.1f} MB')
print('Se descarga una vez en el navegador y despues funciona sin conexion.')
for f in sorted(DIR_MODELO.iterdir()):
    print(f'   {f.name}  ({f.stat().st_size / 1e6:.2f} MB)')

## 15. Generar el archivo de clases para la app

Produce `clases.ts`, que va en `src/lib/vision/`. Contiene el índice de cada
clase, su etiqueta legible, de qué dataset viene y —cuando aplica— el `plagaId`
que enlaza con `catalogo.ts` para mostrar la ficha completa.

In [ ]:
acc_global = float(acierto.mean())
acc_pv = float(acierto[origen == 'plantvillage'].mean()) if (origen == 'plantvillage').any() else 0.0
acc_ip = float(acierto[origen == 'ip102'].mean()) if (origen == 'ip102').any() else 0.0

lineas = [
    '/**',
    ' * @fileOverview Clases del clasificador de imagenes. GENERADO AUTOMATICAMENTE.',
    ' *',
    ' * Lo produce notebooks/entrenar_clasificador.ipynb (celda 15). No lo edites a',
    ' * mano: al reentrenar cambian los indices y se sobrescribe.',
    ' *',
    ' * Desempenyo medido sobre el conjunto de prueba:',
    f' *   global                    {acc_global:.1%}',
    f' *   PlantVillage (laboratorio) {acc_pv:.1%}',
    f' *   IP102 (campo real)         {acc_ip:.1%}',
    ' *',
    ' * El numero global mezcla ambos conjuntos y queda inflado por la parte facil.',
    ' * Al citar resultados, usa siempre los tres.',
    ' */',
    '',
    'export interface ClaseModelo {',
    '  indice: number;',
    '  /** Nombre de la carpeta original del dataset. */',
    '  clase: string;',
    '  /** Texto que se muestra al agricultor. */',
    '  etiqueta: string;',
    "  origen: 'plantvillage' | 'ip102';",
    '  /** id de src/lib/plagas/catalogo.ts, si esta clase tiene ficha. */',
    '  plagaId?: string;',
    '}',
    '',
    '/** Metricas del modelo entrenado, para mostrarlas en la interfaz. */',
    'export const METRICAS_MODELO = {',
    f'  global: {acc_global:.4f},',
    f'  plantvillage: {acc_pv:.4f},',
    f'  ip102: {acc_ip:.4f},',
    f'  totalClases: {len(clases)},',
    '} as const;',
    '',
    'export const CLASES_MODELO: readonly ClaseModelo[] = [',
]

for m in MAPEO:
    etq = m['etiqueta'].replace("'", "\\'")
    cls = m['clase'].replace("'", "\\'")
    campos = [
        f"indice: {m['indice']}",
        f"clase: '{cls}'",
        f"etiqueta: '{etq}'",
        f"origen: '{m['origen']}'",
    ]
    if m['plagaId']:
        campos.append(f"plagaId: '{m['plagaId']}'")
    lineas.append('  { ' + ', '.join(campos) + ' },')

lineas += [
    '] as const;',
    '',
    '/** Busca una clase por su indice de salida del modelo. */',
    'export function claseporIndice(indice: number): ClaseModelo | undefined {',
    '  return CLASES_MODELO[indice];',
    '}',
    '',
]

ruta_ts = SALIDA / 'clases.ts'
ruta_ts.write_text('\n'.join(lineas), encoding='utf-8')
print(f'clases.ts generado ({len(clases)} clases, {len(con_ficha)} con ficha)')

## 16. Descargar todo

Se descarga un `.zip`. Dentro vienen las instrucciones de dónde va cada archivo.

In [ ]:
(SALIDA / 'LEEME.txt').write_text(f"""MODELO ENTRENADO — AgroTech Hidalgo

Resultados sobre el conjunto de prueba:
  global                     {acc_global:.1%}
  PlantVillage (laboratorio) {acc_pv:.1%}
  IP102 (campo real)         {acc_ip:.1%}

  {len(clases)} clases, {len(con_ficha)} enlazadas al catalogo de la app.
  Tamano del modelo: {tam_mb:.1f} MB

DONDE VA CADA COSA

  modelo/     ->  public/modelo/        (en la raiz del proyecto cardenas)
  clases.ts   ->  src/lib/vision/clases.ts

En Windows, desde la carpeta del proyecto:

  Expand-Archive modelo_web.zip -DestinationPath .\\temp_modelo
  New-Item -ItemType Directory -Force public\\modelo
  Copy-Item .\\temp_modelo\\modelo\\* public\\modelo\\ -Recurse -Force
  Copy-Item .\\temp_modelo\\clases.ts src\\lib\\vision\\clases.ts -Force
  Remove-Item .\\temp_modelo -Recurse -Force

AL CITAR RESULTADOS

  Da los tres numeros, no solo el global. PlantVillage son fotos de laboratorio
  sobre fondo liso y el modelo acierta mucho ahi; IP102 son fotos de campo y es
  el numero que de verdad representa el uso real. Decirlo tu mismo, antes de que
  te lo pregunten, es lo que distingue un trabajo serio.
""", encoding='utf-8')

shutil.make_archive('/content/modelo_web', 'zip', str(SALIDA))

from google.colab import files
files.download('/content/modelo_web.zip')

print('\nDescargando modelo_web.zip')
print(f'  {len(clases)} clases | {tam_mb:.1f} MB')
print(f'  laboratorio {acc_pv:.1%} | campo real {acc_ip:.1%}')